# Milestone 2 — Data Preparation

**Goal of this notebook:** turn the raw xLAM-function-calling-60k dataset into a clean, masked, tokenized
`DatasetDict` (train/validation/test) ready to feed into Milestone 3's `SFTTrainer`. By the end you'll have:

1. A converted tool-schema format that matches what Qwen's chat template actually expects (the source
   dataset's format does **not** match this out of the box).
2. A loss-masking scheme that trains only on the assistant's `<tool_call>` tokens, not the prompt.
3. Train (57,064) / validation (1,000) / test (1,000) splits, saved to Google Drive.

**Runtime:** this notebook does not need a GPU — a **CPU-only Colab runtime** is enough and saves your T4
quota for Milestone 3. Everything here should run in a few minutes total, dataset conversion included.

Repo: `github.com/necdetduruk/qwen-function-calling-qlora` — this notebook is
`notebooks/02_data_preparation.ipynb`.

## Step 0 — Install & authenticate

`Salesforce/xlam-function-calling-60k` is a **gated** dataset on the Hugging Face Hub — a valid token isn't
enough on its own, your account also needs to have clicked through the access request on the dataset's page
at least once (`huggingface.co/datasets/Salesforce/xlam-function-calling-60k`, usually auto-approved
instantly). If `load_dataset(...)` below raises `DatasetNotFoundError` mentioning "gated dataset," that's
what's happening — go accept access on that page, then re-run this cell.

In [ ]:
!pip install -q datasets==3.0.2 huggingface_hub==0.26.2

import os

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(name)
    if val:
        return val
    from getpass import getpass
    return getpass(f"Enter {name}: ")

from huggingface_hub import login
login(token=get_secret("HF_TOKEN"), add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## Step 1 — Load and inspect the raw dataset

Load it and look at the raw schema before assuming anything about its shape.

In [ ]:
import json
from datasets import load_dataset

ds = load_dataset("Salesforce/xlam-function-calling-60k")
print(ds)
print(ds["train"][0])

**Expected output:** a `DatasetDict` with a single `train` split, 60,000 rows, features
`['id', 'query', 'answers', 'tools']`. Note there's no pre-made test split — we'll carve our own below.
`answers` and `tools` print as escaped JSON *strings*, not parsed objects yet.

## Step 2 — A readable view (don't just `json.dumps` and squint)

Raw nested JSON is hard to eyeball even pretty-printed. A small formatter that turns each row into a few
plain-English lines makes it much easier to actually notice what's going on structurally.

In [ ]:
def show_example(i):
    ex = ds["train"][i]
    tools = json.loads(ex["tools"])
    answers = json.loads(ex["answers"])

    print(f"QUERY: {ex['query']}")
    print(f"\nTOOLS AVAILABLE ({len(tools)}):")
    for t in tools:
        params = t.get("parameters", {})
        param_str = ", ".join(f"{name}: {info.get('type')}" for name, info in params.items())
        print(f"  - {t['name']}({param_str})")
        print(f"      \"{t['description']}\"")

    print(f"\nEXPECTED CALL(S) ({len(answers)}):")
    for a in answers:
        print(f"  - {a['name']}({a['arguments']})")
    print("\n" + "=" * 70)

for i in [0, 1, 2, 50]:
    show_example(i)

**What this reveals, that matters for everything downstream:**

- **Distractor tools are real** — some rows offer several candidate tools and only 1-2 are actually called.
  Part of the task is correct *selection*, not just formatting.
- **Parallel calls are real** — some rows expect more than one function call for a single query.
- **Data quality noise exists** — e.g. one tool's `description` is literally the string `"python"`. We don't
  hand-clean 60k tool descriptions; this is a documented limitation, not something we silently paper over.

## Step 3 — The schema mismatch: xLAM's tool format vs. Qwen's chat template

Compare an xLAM tool entry to the schema Qwen's chat template actually expects (OpenAI/JSON-Schema style):

```python
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "...",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "..."}},
            "required": ["city"],
        },
    },
}
```

Three concrete differences:

1. xLAM tools are **flat** — no `{"type": "function", "function": {...}}` wrapper.
2. xLAM's `parameters` is a plain `name -> {description, type, default}` dict, not a JSON-Schema object with
   `properties` / `required`.
3. Type names are Python-style (`str`, `int`) instead of JSON-Schema style (`string`, `integer`) — and, as
   the next step shows, they're much messier than that.

There's also no explicit `required` list. The only reliable signal for "this parameter is optional" is the
literal substring `"optional"` inside the `type` field — a `default` value being present is **not** a
reliable signal (some required-looking params still carry a `default`).

## Step 4 — What does the `type` field actually contain, across all 60k rows?

Don't guess the type vocabulary from four examples — scan the whole dataset and get real counts.

In [ ]:
from collections import Counter

raw_type_counts = Counter()

for row in ds["train"]:
    tools = json.loads(row["tools"])
    for tool in tools:
        for param_info in tool["parameters"].values():
            raw_type_counts[param_info.get("type", "<missing>")] += 1

for t, c in raw_type_counts.most_common():
    print(f"{c:>6}  {t}")

**What this shows:** the `type` field isn't a controlled vocabulary — it's raw Python type-hint text
(`List[Tuple[int, int]]`, `Callable[[float], float]`, `str, optional, default='elonmusk'`). `str` / `int` /
`float` / `bool` (with or without `, optional`) dominate overwhelmingly. Flat lists of primitives
(`List[int]`, `List[str]`, etc.) are common enough (~19k instances) to handle properly, not drop. Nested
containers (`Dict`, `Tuple[...]`, `set`, `List[List[...]]`) are rarer but still mappable to JSON `object` /
`array`. **Only `Callable[[float], float]` (593 instances) is a genuine dead end** — a parameter whose value
is supposed to be a Python function object cannot be represented as a JSON argument, period. That's a
task-definition mismatch, not a formatting one.

## Step 5 — Normalize the type strings

One function, built to **fail loudly on anything unrecognized** rather than silently guessing — a silent
`else: return "string"` would quietly mis-type any raw type string we haven't already seen and verified.

In [ ]:
import re

def normalize_param_type(raw_type: str):
    \"\"\"Returns (json_type, items_type_or_None, is_optional, supported: bool).\"\"\"
    is_optional = "optional" in raw_type
    base = raw_type.split(",")[0].strip()  # text before first comma

    scalar_map = {"str": "string", "int": "integer", "float": "number", "bool": "boolean"}
    if base in scalar_map:
        return scalar_map[base], None, is_optional, True

    if base.lower() in ("list", "set", "tuple") or base.startswith(("List", "Tuple")):
        m = re.match(r"List\[(str|int|float)\]$", base)
        items_type = scalar_map.get(m.group(1)) if m else None
        return "array", items_type, is_optional, True

    if base.lower() == "dict" or base.startswith("Dict"):
        return "object", None, is_optional, True

    if base.startswith("Callable"):
        return None, None, is_optional, False  # genuinely unsupported

    return None, None, is_optional, False  # unknown -> fail loudly, don't guess


# Verify against every distinct raw type actually found in the dataset (Step 4), not just examples we
# happened to look at by hand.
for t, c in raw_type_counts.most_common():
    json_type, items_type, is_optional, supported = normalize_param_type(t)
    flag = "OK" if supported else "UNSUPPORTED"
    print(f"{c:>6}  {t!r:45} -> type={json_type}, items={items_type}, optional={is_optional}  [{flag}]")

**Expected output:** every line reads `[OK]` except `Callable[[float], float]`, which reads `[UNSUPPORTED]`.
If anything else shows `UNSUPPORTED`, the mapper missed a case that needs handling before moving on — don't
ignore it.

## Step 6 — Convert tools to Qwen/OpenAI format, and measure the real drop rate

`xlam_tool_to_qwen_format` returns `None` when a tool can't be faithfully represented. We drop an entire
example if **any** tool in its list — called or not — fails to convert, rather than surgically deleting just
the bad tool. Two reasons: (1) distractor tools are part of the actual task signal (the model has to learn
to *reject* irrelevant tools, so quietly changing the distractor set changes the problem), and (2) if the
broken tool happens to be the one that *was* called, surgical repair isn't even possible — you'd need two
different code paths for what should be one simple, verifiable rule.

In [ ]:
def xlam_tool_to_qwen_format(tool):
    \"\"\"Convert one xLAM tool dict to Qwen/OpenAI-style function schema.
    Returns None if any parameter type can't be represented in JSON (e.g. Callable).\"\"\"
    properties = {}
    required = []

    for param_name, param_info in tool["parameters"].items():
        raw_type = param_info.get("type", "str")
        json_type, items_type, is_optional, supported = normalize_param_type(raw_type)

        if not supported:
            return None

        prop = {"type": json_type, "description": param_info.get("description", "")}
        if json_type == "array" and items_type is not None:
            prop["items"] = {"type": items_type}
        properties[param_name] = prop

        if not is_optional:
            required.append(param_name)

    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


clean_examples = []

for row in ds["train"]:
    tools_raw = json.loads(row["tools"])
    converted_tools = [xlam_tool_to_qwen_format(t) for t in tools_raw]

    if any(t is None for t in converted_tools):
        continue

    clean_examples.append({
        "id": row["id"],
        "query": row["query"],
        "tools": converted_tools,
        "answers": json.loads(row["answers"]),
    })

print(f"clean_examples: {len(clean_examples)}  (dropped {60000 - len(clean_examples)}, "
      f"{(60000 - len(clean_examples)) / 60000:.2%})")

**Expected output:** `clean_examples: 59407  (dropped 593, 0.99%)` — a small, defensible loss to guarantee
every remaining tool schema is valid JSON-Schema, with no ambiguous partial-repair logic anywhere in the
pipeline.

## Step 7 — The chat template: building a real training conversation

Loading just the tokenizer here (no 4-bit model, no GPU needed) — `apply_chat_template` lives on the
tokenizer.

**Why not hand-write `f"<tool_call>{json.dumps(call)}</tool_call>"` ourselves instead?** It's not a
readability question — it's a token-level correctness question. `apply_chat_template` renders the *exact*
characters, spacing, and special tokens (`<|im_start|>`, `<|im_end|>`, how multiple `<tool_call>` blocks are
joined) that Qwen's own instruction-tuning used. Tokenizers use byte-pair merges that are sensitive to exact
adjacent characters, so a hand-rolled near-match can tokenize into a *different* sequence than the official
template would, even though the text looks identical when printed. This fails **silently** — the loss still
goes down, nothing crashes — and shows up later as a train/inference mismatch that's brutal to debug, since
we build prompts with `apply_chat_template` again at inference time.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

example = clean_examples[0]

messages = [
    {"role": "user", "content": example["query"]},
    {
        "role": "assistant",
        "tool_calls": [
            {"type": "function", "function": {"name": call["name"], "arguments": call["arguments"]}}
            for call in example["answers"]
        ],
    },
]

rendered = tokenizer.apply_chat_template(
    messages,
    tools=example["tools"],
    tokenize=False,
    add_generation_prompt=False,  # we're supplying the full assistant answer ourselves
)
print(rendered)

**Expected output:** `<|im_start|>system` with a `<tools>` block listing our converted schema, an
`<|im_start|>user` turn with the query, then `<|im_start|>assistant` containing one `<tool_call>...</tool_call>`
block per answer, ending `<|im_end|>`.

## Step 8 — Loss masking: only train on the assistant's answer

If we computed loss over the *entire* tokenized sequence, the model would be penalized for not perfectly
"predicting" the system boilerplate and the user's query — text it should only be reading, not generating.
Worse, the prompt (system + tools + user) is typically far longer than the actual answer, so unmasked
training would spend most of its gradient budget reconstructing input instead of learning the one skill we
actually care about.

The fix: tokenize the full conversation and a prompt-only version (same messages minus the assistant turn,
with `add_generation_prompt=True` so it ends exactly where the assistant's turn begins), then set every
prompt-token position to `-100` (PyTorch's `CrossEntropyLoss` "ignore this position" value) and leave real
token IDs only where the assistant's answer is.

**A real correctness risk here:** tokenizing a prompt alone can, in principle, produce slightly different
tokens right at the boundary than tokenizing it as a prefix of the full text (BPE merges are
context-sensitive). We verify the prefix assumption explicitly instead of trusting it.

In [ ]:
def build_training_example(example, tokenizer):
    messages = [
        {"role": "user", "content": example["query"]},
        {
            "role": "assistant",
            "tool_calls": [
                {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                for c in example["answers"]
            ],
        },
    ]

    full_text = tokenizer.apply_chat_template(
        messages, tools=example["tools"], tokenize=False, add_generation_prompt=False
    )
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1], tools=example["tools"], tokenize=False, add_generation_prompt=True
    )

    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Prompt is not a clean prefix for example id={example['id']}")

    labels = full_ids.copy()
    labels[: len(prompt_ids)] = [-100] * len(prompt_ids)

    return {"input_ids": full_ids, "labels": labels}


# Test on a harder case: 7 candidate tools, 2 expected calls -- not just the easy single-tool example.
test_example = next(e for e in clean_examples if e["id"] == 50)
built = build_training_example(test_example, tokenizer)

print("total tokens:", len(built["input_ids"]))
print("trainable tokens:", sum(1 for l in built["labels"] if l != -100))
print("\ntrainable text:\n")
print(tokenizer.decode([t for t, l in zip(built["input_ids"], built["labels"]) if l != -100]))

**Expected output:** `total tokens: 1097`, `trainable tokens: 81` — only ~7% of the sequence is actually
trainable, a vivid illustration of why masking matters. The decoded trainable text should show exactly the
two expected `<tool_call>` blocks, nothing from the other 5 distractor tool schemas or the query.

## Step 9 — Apply to the full dataset, and check sequence lengths

We tokenize all 59,407 examples, catching (not crashing on) any per-row failure so one bad row can't waste
several minutes of work. We collect token lengths in the same pass, since we'll need them immediately below
to choose `max_seq_length`.

In [ ]:
built_examples = []
failed_ids = []

for ex in clean_examples:
    try:
        built = build_training_example(ex, tokenizer)
        built["id"] = ex["id"]
        built_examples.append(built)
    except ValueError:
        failed_ids.append(ex["id"])

print(f"built: {len(built_examples)}  failed: {len(failed_ids)}")
if failed_ids:
    print("failed ids (first 10):", failed_ids[:10])

In [ ]:
import numpy as np

lengths = np.array([len(b["input_ids"]) for b in built_examples])
print("min:", lengths.min())
print("max:", lengths.max())
print("mean:", lengths.mean())
for p in [50, 90, 95, 99]:
    print(f"p{p}:", np.percentile(lengths, p))

**Expected:** `built: 59407  failed: 0` (our earlier filtering already removed the only structural failure
mode, `Callable`), and a length distribution with `p50` ~500, `p95` ~964, `p99` ~1200, `max` ~2553 — most
examples cluster in a few hundred tokens, with a thin long tail from examples with many candidate tools.

## Step 10 — Choosing `max_seq_length`: drop, don't truncate

Right-truncating a sequence to fit `max_seq_length` is risky here in a way that isn't obvious at first: if a
row's **prompt alone** (system + tools + user) already exceeds `max_seq_length`, truncation silently deletes
the entire answer — the row trains on nothing (all labels `-100`) with no error to say so. And even a row
that keeps *some* of its answer could have it cut off mid-JSON, teaching the model that truncated, invalid
JSON is an acceptable output. Both are silent correctness bugs hiding behind a single config number.

**Decision: `max_seq_length = 1280`, and drop every example that exceeds it** (not just the ones that would
lose their answer entirely). This guarantees every kept example fits completely — no truncation-aware logic
needed anywhere downstream, and no way for either failure mode to exist. Checked against real counts: at
1280, only 343 of 59,407 examples (0.58%) exceed it — a small, worthwhile price for eliminating the whole
failure class.

In [ ]:
MAX_SEQ_LENGTH = 1280

final_examples = [b for b in built_examples if len(b["input_ids"]) <= MAX_SEQ_LENGTH]
print(f"final_examples: {len(final_examples)} (dropped {len(built_examples) - len(final_examples)})")

**Expected output:** `final_examples: 59064 (dropped 343)`.

## Step 11 — Train / validation / test splits, with an explicit leakage check

1,000 validation / 1,000 test / everything else for training. Round numbers are easy to reason about, and
1,000 rows is enough for statistically meaningful JSON-validity / exact-match rates in Milestone 4 without
giving up much training data. We shuffle once with a fixed seed (reproducibility) and then **explicitly
verify zero overlap** between splits — silent train/test leakage is one of the most common, hardest-to-spot
bugs in ML pipelines, and "the slicing looked right" isn't the same as "we checked."

In [ ]:
import random

random.seed(42)
shuffled = final_examples[:]
random.shuffle(shuffled)

N_TEST = 1000
N_VAL = 1000

test_set = shuffled[:N_TEST]
val_set = shuffled[N_TEST:N_TEST + N_VAL]
train_set = shuffled[N_TEST + N_VAL:]

print(f"train: {len(train_set)}  val: {len(val_set)}  test: {len(test_set)}")

train_ids = {e["id"] for e in train_set}
val_ids = {e["id"] for e in val_set}
test_ids = {e["id"] for e in test_set}

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
print("No overlap between splits — confirmed.")

**Expected output:** `train: 57064  val: 1000  test: 1000`, followed by the confirmation line. If any
`assert` fails, stop — that means the slicing logic is wrong, not the data.

## Step 12 — Hand-inspected sample (don't skip this)

Counts and length stats can't catch every bug. Pull a few finished examples from the *final* processed test
set and read them end to end: is the trainable portion always a complete, valid `<tool_call>` block (or
blocks) followed by `<|im_end|>`? Is the right tool picked out of the distractors? Do argument types (string
vs. integer) look right given the type-normalization work in Step 5?

In [ ]:
import random as _random

_random.seed(0)  # separate seed, just for picking which examples to display
sample_indices = _random.sample(range(len(test_set)), 3)

for i in sample_indices:
    ex = test_set[i]
    full_text = tokenizer.decode(ex["input_ids"])
    trainable_text = tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])

    print("=" * 80)
    print(f"example id: {ex['id']}")
    print("-" * 80)
    print("FULL CONVERSATION:")
    print(full_text)
    print("-" * 80)
    print("TRAINABLE PORTION ONLY (what the loss actually sees):")
    print(trainable_text)
    print()

**What to check by eye:** complete, valid JSON in every `<tool_call>` block; correct tool selected among the
distractors; argument types matching what Step 5 assigned (e.g. an `integer`-typed argument rendering as a
real number, not a quoted string); zero leaked prompt text in the trainable portion.

## Step 13 — Persist to Google Drive

Colab's local disk disappears when the runtime recycles; Drive doesn't. Saving here means Milestone 3 — in
this session or a fresh one — just mounts Drive and calls `DatasetDict.load_from_disk(...)`, with no need to
re-download the gated dataset or re-run several minutes of tokenization. This is also why `data/processed/`
is already in the repo's `.gitignore` — this is regeneratable, sizeable derived data; Drive is the right
home for it, git isn't.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datasets import Dataset, DatasetDict

def to_hf_dataset(examples):
    return Dataset.from_dict({
        "id": [e["id"] for e in examples],
        "input_ids": [e["input_ids"] for e in examples],
        "labels": [e["labels"] for e in examples],
    })

dataset_dict = DatasetDict({
    "train": to_hf_dataset(train_set),
    "validation": to_hf_dataset(val_set),
    "test": to_hf_dataset(test_set),
})

save_dir = "/content/drive/MyDrive/qwen-function-calling-qlora/data/processed"
os.makedirs(save_dir, exist_ok=True)
dataset_dict.save_to_disk(save_dir)

print(dataset_dict)
print(f"Saved to: {save_dir}")

**Expected output:** a `DatasetDict` summary (`train`: 57064 rows, `validation`: 1000, `test`: 1000, each
with features `['id', 'input_ids', 'labels']`) and the save-path confirmation.

## Checkpoint — commit this

```bash
git add notebooks/02_data_preparation.ipynb
git commit -m "M2: data prep - xLAM conversion, loss masking, train/val/test splits"
git push
```

The processed dataset itself (`data/processed/`) stays out of git by design (see `.gitignore`) — it's
regeneratable from this notebook plus the Drive copy, and it's too large/derived to belong in version
control. What we commit is the *pipeline*, not its output.

## Before you move on — answer these (Socratic check)

1. We drop an entire example if **any** tool in its list has an unsupported type, even a distractor tool
   that's never called. Why is that simpler and safer than surgically deleting just the bad tool and keeping
   the rest of the example?
2. What would silently go wrong if we trained without loss masking — i.e., computed loss over the system
   message, tool list, and user query too, not just the assistant's answer?
3. `apply_chat_template` renders the exact tokens Qwen's own instruction-tuning used. What specifically could
   go wrong, at the token level, if we hand-rolled a prompt string that merely *looked* like the same format?
4. We chose to drop examples exceeding `max_seq_length` rather than truncate them. What specific failure mode
   does truncation risk here that a simple length filter avoids entirely?

In [ ]:
# Your answers here (or in a markdown cell above) -- we'll revisit anything marked TODO in the interview
# prep pass (Milestone 8).

answer_1 = "Distractor tools are part of the actual tool-selection task, and a broken tool could be the " \
           "one actually called (in which case surgical repair isn't even possible) -- dropping the whole " \
           "row is one simple rule instead of two fragile special cases, for a <1% data cost."
answer_2 = "The model would waste gradient trying to 'predict' the system message and the user's query -- " \
           "text that's genuinely unpredictable to read forward over (there's no 'right' next word for an " \
           "arbitrary question) -- and since the prompt is far longer than the answer, most of the training " \
           "signal would go toward reconstructing input instead of learning to call functions correctly."
answer_3 = "TODO -- revisit in Milestone 8. (Token-level mismatch is the core idea: BPE merges are sensitive " \
           "to exact characters/spacing, so a hand-rolled near-match can tokenize differently than the " \
           "official template even if the text looks identical -- and it fails silently, showing up later " \
           "as train/inference skew. Practice explaining this one out loud before the interview prep pass.)"
answer_4 = "Truncating could delete the entire answer (if the prompt alone exceeds max_seq_length) or cut " \
           "it off mid-JSON, teaching the model that truncated/invalid output is acceptable -- both silent " \
           "bugs a length filter avoids since every kept example is guaranteed complete."